# RF Análisis Fase 5.0 — Sionna Ray Tracing (DATA REAL)

**Versión 5.0** · usa el modelo **real** de UP9 derivado de los planos DWG entregados por SPF Santa Fe (2026-04-28).

**Cambios vs v4.12** (sintético):
- 57 edificios reales (vs 20 sintéticos a ojo)
- Geometrías precisas (polígonos multi-vértice del DWG, no rectángulos de 4 esquinas)
- Centro UP9 actualizado al validado (-31.510037, -60.721543 vs -31.509878, -60.721272)
- Filtrado a SU1 (sub-unidad construida) — las otras 3 SUs del master plan no existen físicamente
- Fuente: `up9_buildings_s1_calado.json` (calibrado manualmente sobre satélite Google Maps)

**Pipeline DXF→JSON→Sionna**:
1. DWG → DXF (ODA File Converter)
2. DXF → footprints atómicos (ezdxf + shapely buffer+union, filtro radio 45m)
3. Footprints → SU1 only + coords locales (script `10_export_su1_local.py`)
4. Coords locales → GPS via editor interactivo Leaflet (`editor_georef.html`)
5. JSON GPS → este notebook → Sionna RT → heatmaps reales


## 1 · Setup: Install Sionna + dependencies

In [ ]:
# =============================================================================
# NOTEBOOK VERSION: v4.12  (17-abr-2026)  [v4.12: calibración dual LOS/NLOS (two-slope ITU-R M.2412)]
# Fixes: v4.1a (earcut), v4.1b (BSDF null), v4.2 (broadband mats), v4.3 (remove ITU), v4.4 (idempotente), v4.5 (drjit Float cast)
# =============================================================================
print('=' * 70)
print('  NOTEBOOK VERSION: v4.12')
print('  Fixes: earcut + BSDF null + broadband mats + remove ITU registry')
print('=' * 70)

# Instalación (~ 2-3 min en Colab). Comentar si ya instalado.
# NOTA: usamos sionna-rt (standalone) porque sionna 0.19.1 pedía TF<2.16
# que no tiene wheels para Python 3.12 (default actual de Colab).
!pip install -q sionna-rt
!pip install -q pyproj shapely trimesh mapbox-earcut
!pip install -q --upgrade matplotlib pandas

# Check GPU + versions + API introspection
import subprocess, sys, inspect
gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode().strip()
print('GPU:', gpu)
print('Python:', sys.version.split()[0])

import sionna.rt as rt
print('sionna-rt version:', getattr(rt, '__version__', '?'))

# Importar símbolos que usaremos — si falla alguno, se ve ya en la celda 1
from sionna.rt import load_scene, Transmitter, Receiver, PlanarArray, RadioMapSolver
print('Imports: load_scene, Transmitter, Receiver, PlanarArray, RadioMapSolver OK')

# Diagnóstico de signatures (para detectar drift de API antes de llegar a la celda 8)
print('\n--- Transmitter.__init__ signature ---')
print(inspect.signature(Transmitter.__init__))
print('\n--- PlanarArray.__init__ signature ---')
print(inspect.signature(PlanarArray.__init__))
print('\n--- RadioMapSolver.__call__ signature ---')
try:
    print(inspect.signature(RadioMapSolver.__call__))
except (ValueError, TypeError) as e:
    print(f'(could not introspect: {e})')

# Verificar triangulation engine para trimesh (crítico para celda 5)
import trimesh
try:
    import mapbox_earcut
    print('\nmapbox_earcut OK — trimesh podrá triangular polígonos')
except ImportError:
    print('\n⚠️  mapbox_earcut NO disponible — celda 5 va a skipear edificios!')


## 2 · Setup repo + inputs

Este notebook clona el repo público `MollytheCatLoca/RFQ_MOTOROLA_IMSI` y carga inputs desde `recreo_v5_real/inputs/`.

**No requiere Google Drive**. Si querés usar Drive (alternative para outputs persistentes), podés montarlo a parte.


In [ ]:
# Clone repo (inputs + notebook) — alternativa a Drive mount
import os
from pathlib import Path

REPO_DIR = Path('/content/RFQ_MOTOROLA_IMSI')
if not REPO_DIR.exists():
    !git clone --depth=1 https://github.com/MollytheCatLoca/RFQ_MOTOROLA_IMSI.git /content/RFQ_MOTOROLA_IMSI
else:
    !cd /content/RFQ_MOTOROLA_IMSI && git pull --rebase

DRIVE_BASE = REPO_DIR / 'recreo_v5_real' / 'inputs'
OUTPUTS_DIR = REPO_DIR / 'recreo_v5_real' / 'outputs'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'DRIVE_BASE = {DRIVE_BASE}')
print(f'OUTPUTS_DIR = {OUTPUTS_DIR}')
import os
for f in sorted(os.listdir(DRIVE_BASE)):
    size_kb = os.path.getsize(DRIVE_BASE / f) / 1024
    print(f'  {f:55s}  {size_kb:>8.1f} KB')

# Standard Python imports needed by next cells
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 3 · Load datasets

In [ ]:
# Edificios — DATA REAL (57 edificios SU1 calibrados sobre satélite)
with open(DRIVE_BASE / 'up9_buildings_s1_calado.json') as f:
    BUILDINGS = json.load(f)['buildings']
print(f'Edificios reales: {len(BUILDINGS)}')

# Verificar tipos
from collections import Counter
type_counts = Counter(b['tipo'] for b in BUILDINGS)
print(f'Por tipo: {dict(type_counts)}')

# Antenas telco (sin cambios — mismo dataset)
ANTENAS = pd.read_csv(DRIVE_BASE / 'antenas_recreo_enriched.csv')
print(f'Antenas: {len(ANTENAS)}')

# IDR ground truth (sin cambios)
IDR = pd.read_csv(DRIVE_BASE / 'mediciones_raw.csv')
IDR = IDR[IDR['is_usable_for_rf_model'] == True]
print(f'Markers IDR usables: {len(IDR)}')

# Sample first building
b = BUILDINGS[0]
print(f'\nSample: {b["name"]} | tipo={b["tipo"]} | altura={b["altura_m"]}m | muros={b["muro_db"]}dB | corners={len(b["corners"])}')


## 4 · Coordenadas locales (ENU meters)

Sionna trabaja en coordenadas cartesianas locales (metros). Convertimos lat/lon al sistema ENU (East-North-Up) con origen en el centro del polígono UP9.


In [ ]:
# Centro UP9 — REAL calibrado (anchor del editor manual sobre satélite)
ORIGIN_LAT = -31.51003689690043   # antes: -31.509878 (MS3 estimado)
ORIGIN_LON = -60.72154283523560   # antes: -60.721272
M_PER_DEG_LAT = 111320.0
M_PER_DEG_LON = 111320.0 * math.cos(math.radians(ORIGIN_LAT))

def to_enu(lat, lon, altitude=0.0):
    east = (lon - ORIGIN_LON) * M_PER_DEG_LON
    north = (lat - ORIGIN_LAT) * M_PER_DEG_LAT
    return east, north, altitude

def to_latlon(east, north):
    lat = ORIGIN_LAT + north / M_PER_DEG_LAT
    lon = ORIGIN_LON + east / M_PER_DEG_LON
    return lat, lon

# Test: 3 puntos IDR
for lat, lon, label in [(-31.509889, -60.718900, 'MS1 back'),
                         (-31.509433, -60.724133, 'MS2 front'),
                         (-31.509878, -60.721272, 'MS3 interior (anterior origen)')]:
    e, n, _ = to_enu(lat, lon)
    print(f'{label:35s}: E={e:+7.1f}m  N={n:+7.1f}m')

# Bbox del nuevo dataset
all_lats = [c[0] for b in BUILDINGS for c in b['corners']]
all_lons = [c[1] for b in BUILDINGS for c in b['corners']]
e_min, n_min, _ = to_enu(min(all_lats), min(all_lons))
e_max, n_max, _ = to_enu(max(all_lats), max(all_lons))
print(f'\nBbox edificios reales: E=[{e_min:.0f},{e_max:.0f}]m  N=[{n_min:.0f},{n_max:.0f}]m')
print(f'Tamaño: {e_max-e_min:.0f} x {n_max-n_min:.0f} m')


## 5 · Build scene — Mitsuba XML

Genera la escena 3D:
- **Ground plane**: 2km × 2km (cubre todas las antenas relevantes)
- **UP9 polygon**: perímetro del penal (piso de referencia)
- **Buildings**: los 20 polígonos extruidos a su `altura_m`

Materiales simplificados (propiedades RF Sionna):
- `concrete`: paredes de pabellones
- `ground`: terreno (tierra compacta + hierba)


In [ ]:
# Generación programática de meshes (buildings + ground)
# FIX v4.1: engine="earcut" explícito (trimesh por default usa "triangle" que no está en Colab)
import trimesh
from shapely.geometry import Polygon as ShapelyPolygon

SCENE_DIR = Path('/content/scene')
SCENE_DIR.mkdir(exist_ok=True)

# Sanity check — triangulation engine
try:
    import mapbox_earcut
    _TRI_ENGINE = 'earcut'
    print('✅ mapbox_earcut OK → engine=earcut')
except ImportError:
    raise RuntimeError('mapbox_earcut no instalado. Correr !pip install mapbox-earcut y restart session.')

# Validación de prerequisitos (definidos en celdas 3 y 4)
assert 'BUILDINGS' in globals() and len(BUILDINGS) > 0, 'BUILDINGS vacío — revisar celda 3'
assert 'to_enu' in globals(), 'to_enu no definido — revisar celda 4'
print(f'Input: {len(BUILDINGS)} edificios a procesar')


def polygon_to_prism_mesh(corners_latlon, height_m, name='building'):
    """Extrude a 2D polygon (lat/lon corners) into a 3D prism mesh."""
    # Convert to local ENU 2D
    pts_2d = np.array([to_enu(c[0], c[1])[:2] for c in corners_latlon])

    # Construir Shapely Polygon (forzar orientation correcta)
    poly = ShapelyPolygon(pts_2d)
    if not poly.is_valid:
        # Auto-fix auto-intersecting polygons
        poly = poly.buffer(0)
    if poly.is_empty:
        raise ValueError(f'{name}: polígono vacío tras validación')

    # Extrude con engine explícito — evita el default 'triangle' que no está en Colab
    mesh = trimesh.creation.extrude_polygon(poly, height=height_m, engine=_TRI_ENGINE)

    if mesh is None or len(mesh.vertices) == 0:
        raise ValueError(f'{name}: extrude_polygon devolvió mesh vacío')

    mesh.metadata['name'] = name
    return mesh


# Ground plane (2km × 2km flat, z ≈ 0)
ground = trimesh.creation.box(extents=[2000.0, 2000.0, 0.1])
ground.apply_translation([0.0, 0.0, -0.05])
ground.export(str(SCENE_DIR / 'ground.ply'))
print(f'✅ Ground: 2000×2000 m → ground.ply ({len(ground.vertices)} verts)')

# Buildings
building_meshes = []
errors = []
for b in BUILDINGS:
    try:
        m = polygon_to_prism_mesh(b['corners'], b['altura_m'], name=b['name'])
        building_meshes.append((b, m))
        m.export(str(SCENE_DIR / f"building_{b['id']}.ply"))
    except Exception as e:
        errors.append(f"{b['name']}: {type(e).__name__}: {e}")

print(f'\nBuildings: {len(building_meshes)} / {len(BUILDINGS)} meshes exportados')
if errors:
    print(f'\nErrores ({len(errors)}):')
    for err in errors:
        print(f'  ❌ {err}')

if len(building_meshes) == 0:
    raise RuntimeError('Ningún edificio generado — revisar inputs y engine de triangulación.')

# Resumen
total_verts = sum(len(m.vertices) for _, m in building_meshes)
total_faces = sum(len(m.faces) for _, m in building_meshes)
print(f'\nTotal geometría: {total_verts} vértices, {total_faces} caras')


In [ ]:
# Generate Mitsuba XML
# NOTE: Sionna accepts XML in Mitsuba format. Reference: Sionna RT tutorials.
# FIX v4.1: declaramos BSDFs con type="null" — Sionna RT 1.x los reemplaza
# automáticamente con los RadioMaterial ITU equivalentes por nombre (prefijo "itu_").

xml_parts = ['<?xml version="1.0" encoding="utf-8"?>',
             '<scene version="2.1.0">',
             '  <integrator type="path"/>',
             '',
             '  <!-- BSDF placeholders — Sionna RT sustituye por RadioMaterial ITU -->',
             '  <bsdf type="null" id="mat-itu_concrete"/>',
             '  <bsdf type="null" id="mat-itu_wet_ground"/>',
             '  <bsdf type="null" id="mat-itu_metal"/>',
             '  <bsdf type="null" id="mat-itu_brick"/>',
             '',
             '  <!-- Ground -->',
             '  <shape type="ply" id="ground">',
             f'    <string name="filename" value="{SCENE_DIR}/ground.ply"/>',
             '    <ref name="bsdf" id="mat-itu_wet_ground"/>',
             '  </shape>']

# Buildings as shape nodes
for b, _ in building_meshes:
    xml_parts.append(f'  <!-- {b["name"]} ({b["tipo"]}, {b["altura_m"]}m, muro={b["muro_db"]}dB) -->')
    xml_parts.append(f'  <shape type="ply" id="b_{b["id"]}">')
    xml_parts.append(f'    <string name="filename" value="{SCENE_DIR}/building_{b["id"]}.ply"/>')
    # Asignación por tipo de muro — todos concrete por default; sub-clases pueden variar.
    # Sionna RT 1.x reconoce el id tras el prefijo "mat-" como nombre de RadioMaterial.
    xml_parts.append(f'    <ref name="bsdf" id="mat-itu_concrete"/>')
    xml_parts.append(f'  </shape>')

xml_parts.append('</scene>')
xml_text = '\n'.join(xml_parts)

scene_xml_path = SCENE_DIR / 'up9_scene.xml'
scene_xml_path.write_text(xml_text)
print(f'Scene XML: {scene_xml_path}')
print(f'  {len(xml_parts)} lines')
print(f'  BSDFs declarados: mat-itu_concrete, mat-itu_wet_ground, mat-itu_metal, mat-itu_brick')


## 6 · Load scene in Sionna

In [ ]:
import sionna.rt as rt
from sionna.rt import load_scene, Transmitter, Receiver, PlanarArray, RadioMapSolver

# Load scene
try:
    scene = load_scene(str(scene_xml_path))
    print(f'Scene loaded OK. Objects: {len(scene.objects)}')
except Exception as e:
    print(f'Error loading scene: {e}')
    print('Fallback: using Sionna built-in munich scene for validation')
    scene = load_scene(rt.scene.munich)

# Array configuration — isotropic for telco macros (simplified).
# Sionna 1.x: vertical_spacing/horizontal_spacing ya no se pasan (defaults 0.5λ).
scene.tx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.rx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')


In [ ]:
# FIX v4.10: scattering_coefficient=0.05 en materiales broadband
# v4.9 con diffuse=True sobreestimaba RSS ~35-45 dB porque default scattering_coeff es alto.
# Bajar a 0.05 conserva caminos NLOS pero reduce acumulación de energía scattered.
from sionna.rt import RadioMaterial

def get_or_create_material(scene, name, **kwargs):
    """Idempotente: reusa si ya existe, sino crea."""
    if name in scene.radio_materials:
        # Actualizar scattering_coefficient si cambió (para re-run sin factory reset)
        mat = scene.radio_materials[name]
        if 'scattering_coefficient' in kwargs:
            mat.scattering_coefficient = kwargs['scattering_coefficient']
        return mat
    mat = RadioMaterial(name=name, **kwargs)
    scene.add(mat)
    return mat

# Broadband materials (ITU-R P.2040-3 @ 1 GHz) + scattering_coefficient=0.05
concrete_bb = get_or_create_material(
    scene, "concrete_bb",
    relative_permittivity=5.24,
    conductivity=0.0462,
    scattering_coefficient=0.05,   # ← bajo (vs default ~0.3)
)

wet_ground_bb = get_or_create_material(
    scene, "wet_ground_bb",
    relative_permittivity=30.0,
    conductivity=0.15,
    scattering_coefficient=0.05,
)

# Reasignar objetos
reassigned = 0
for obj_id, obj in scene.objects.items():
    if obj_id == 'ground':
        obj.radio_material = wet_ground_bb
    else:
        obj.radio_material = concrete_bb
    reassigned += 1

# Remover materiales ITU del registro
itu_names = [n for n in list(scene.radio_materials.keys()) if n.startswith('itu_')]
removed, failed = [], []
for name in itu_names:
    try:
        scene.remove(name); removed.append(name)
    except Exception as e:
        failed.append(f'{name}: {e}')

print(f'✅ {reassigned} objetos reasignados (concrete_bb scat=0.05, wet_ground_bb scat=0.05)')
print(f'✅ {len(removed)} ITU removidos: {removed}')
print(f'   Materiales vivos: {list(scene.radio_materials.keys())}')


## 7 · Configure transmitters (antenas telco) + carrier frequency

Cargamos las **top-N antenas más cercanas al UP9** como transmitters. Limitamos N para que el ray-tracing sea tratable (N × bands × receivers puede crecer rápido).


In [ ]:
# Select top-N antennas per band
TOP_N_ANTENNAS = 15  # por banda — ajustar según performance

# Band metadata
BAND_FREQ_MHZ = {
    'B28': 780.5, 'B26': 876.5, 'B41': 2593.0, 'B66': 2155.0,
    'B4': 2132.5, 'B25': 1962.5, 'B2': 1960.0, 'B7': 2655.0,
}
BAND_TO_TECH = {
    'B28': ['LTE'], 'B26': ['LTE', 'UMTS', 'GSM'], 'B41': ['LTE'],
    'B66': ['LTE'], 'B4': ['LTE'], 'B25': ['LTE'],
    'B2': ['LTE'], 'B7': ['LTE'],
}
EIRP_DBM = 45.0  # uniform for simplicity

H_TX = 25.0  # typical macro height (m)

def antennas_for_band(banda):
    techs = BAND_TO_TECH[banda]
    df = ANTENAS[ANTENAS['radio'].isin(techs)].copy()
    df = df.sort_values('distance_km_to_up9').head(TOP_N_ANTENNAS)
    return df

def configure_transmitters(banda):
    '''Clear scene transmitters and add top-N for the given band.'''
    # Remove existing transmitters
    for name in list(scene.transmitters.keys()):
        scene.remove(name)
    df = antennas_for_band(banda)
    for _, row in df.iterrows():
        e, n, _ = to_enu(row['lat'], row['lon'])
        # Sionna 1.x: Transmitter acepta power_dbm directamente
        tx = Transmitter(name=f"tx_{row['id']}",
                         position=[e, n, H_TX],
                         power_dbm=EIRP_DBM)
        scene.add(tx)
    scene.frequency = BAND_FREQ_MHZ[banda] * 1e6
    return len(df)

# Preview
for b in ['B28', 'B26', 'B41']:
    n = configure_transmitters(b)
    print(f'{b}: {n} transmitters @ {BAND_FREQ_MHZ[b]:.0f} MHz')


## 8 · Compute coverage map per banda

Coverage map = raster 2D de potencia recibida sobre el plano Z = altura del receptor.
Corremos 2 capas:
- **z=1.5m** (floor level — equivalente a IDR)
- **z=7.5m** (altura torreón)


In [ ]:
# Sionna 1.x: usamos RadioMapSolver (reemplaza compute_coverage_map).
# Instanciamos una vez y lo invocamos por banda/altura.
rm_solver = RadioMapSolver()

# Diagnóstico: imprimir signature real del solver (por si hay drift de API)
import inspect
try:
    sig = inspect.signature(rm_solver.__call__)
    print('RadioMapSolver.__call__ signature:')
    for name, param in sig.parameters.items():
        print(f'  {name}: {param}')
except Exception as e:
    print(f'Could not introspect: {e}')

# Coverage map bounds — cubrir UP9 + buffer 50m
UP9_CORNERS = [
    (-31.51077550247418, -60.719005465507514),
    (-31.509905410776035, -60.71872919797898),
    (-31.508579105612537, -60.723972916603095),
    (-31.50945035302194,  -60.72424247860909),
]
corners_enu = [to_enu(c[0], c[1])[:2] for c in UP9_CORNERS]
e_min = min(c[0] for c in corners_enu) - 50
e_max = max(c[0] for c in corners_enu) + 50
n_min = min(c[1] for c in corners_enu) - 50
n_max = max(c[1] for c in corners_enu) + 50

print(f'Coverage area: E[{e_min:.0f}, {e_max:.0f}]  N[{n_min:.0f}, {n_max:.0f}]')
print(f'Size: {e_max-e_min:.0f}m × {n_max-n_min:.0f}m')

# Grid resolution (adjust based on GPU memory)
CELL_SIZE_M = 2.0
RESOLUTION = (int((e_max-e_min)/CELL_SIZE_M), int((n_max-n_min)/CELL_SIZE_M))
print(f'Grid resolution: {RESOLUTION[0]} × {RESOLUTION[1]} cells ({RESOLUTION[0]*RESOLUTION[1]} total)')


### 8a · Test run (1 banda × 1 altura) para validar API del solver

Antes de gastar 10-20 min en el full run de 6 combinaciones, corremos una sola con grid reducido.
Si este call falla, pegame el traceback y ajustamos los parámetros de RadioMapSolver sin perder tiempo.


In [ ]:
# TEST RUN — ~30 seg
configure_transmitters('B28')
try:
    rm_test = rm_solver(
        scene=scene,
        max_depth=3,
        cell_size=(CELL_SIZE_M*2, CELL_SIZE_M*2),  # grid 2x más grueso
        center=((e_min+e_max)/2, (n_min+n_max)/2, 1.5),
        orientation=(0, 0, 0),
        size=(e_max-e_min, n_max-n_min),
        samples_per_tx=int(1e5),  # 10x menos muestras
        refraction=True,
        specular_reflection=True,
        diffuse_reflection=False,
    )
    print(f'✅ Test run OK. RadioMap type: {type(rm_test).__name__}')
    print(f'   Attributes (first 15): {[a for a in dir(rm_test) if not a.startswith("_")][:15]}')
    for attr in ['path_gain', 'rss', 'sinr']:
        if hasattr(rm_test, attr):
            val = getattr(rm_test, attr)
            shape = val.shape if hasattr(val, 'shape') else '?'
            print(f'   .{attr}: shape={shape}')
except TypeError as e:
    print(f'❌ API mismatch: {e}')
    print('\n--- Signature real ---')
    print(inspect.signature(rm_solver.__call__))
    raise


### 8b · Full run (6 maps)

In [ ]:
# FIX v4.9: max_depth=7 + diffuse=True + 2e6 samples → intento rescatar MS3 outliers
# v4.8 (max_depth=5, diffraction=True) rescató 6/8 MS con MAE 15.8 dB.
# MS3 B28/B26 siguieron clampeadas → NLOS muy profundo requiere más depth + scattering.

BANDS_TO_COMPUTE = ['B28', 'B26', 'B41']
Z_LEVELS = [1.5, 7.5]

coverage_results = {}

for banda in BANDS_TO_COMPUTE:
    configure_transmitters(banda)
    for z in Z_LEVELS:
        print(f'\n▶ Computing: {banda} @ z={z}m (tx={len(scene.transmitters)})')
        rm = rm_solver(
            scene=scene,
            max_depth=7,                  # +2 niveles (vs 5) → penetración multi-pared
            cell_size=(CELL_SIZE_M, CELL_SIZE_M),
            center=((e_min+e_max)/2, (n_min+n_max)/2, z),
            orientation=(0, 0, 0),
            size=(e_max-e_min, n_max-n_min),
            samples_per_tx=int(2e6),      # 2x rayos para encontrar paths NLOS raros
            refraction=True,
            specular_reflection=True,
            diffuse_reflection=True,      # ← NUEVO: scattering en muros (indoor deep)
            diffraction=True,
            edge_diffraction=True,
        )
        coverage_results[(banda, z)] = rm
        try:
            print(f'  path_gain shape: {rm.path_gain.shape}')
        except AttributeError:
            print(f'  RadioMap attrs: {[a for a in dir(rm) if not a.startswith("_")][:10]}')


## 9 · Export heatmaps + CSV per punto

In [ ]:
# FIX v4.7: cm_to_dbm() agrega correctamente sobre tx (MAX = best server por celda)
# Bug v4.6: rm.rss con shape (15,173,312) no se squeezeaba (no hay dim=1) → imshow de 3D → plot vacío
import matplotlib.colors as mcolors

def _to_numpy(arr):
    """Convert tensor (drjit / tf / np) to numpy array robustly."""
    if hasattr(arr, 'numpy'):
        return arr.numpy()
    return np.array(arr)

def cm_to_dbm(rm, eirp_dbm):
    """Convert Sionna RadioMap to Rx power in dBm — MAX per cell (best server).
    Shape típica en Sionna RT 2.x: (num_tx, ny, nx). Agregamos sobre axis=0.
    """
    if hasattr(rm, 'rss') and rm.rss is not None:
        rss = _to_numpy(rm.rss)
        if rss.ndim == 3:
            rss_2d = np.max(rss, axis=0)  # best server (IMSI detecta el más fuerte)
        else:
            rss_2d = rss
        rss_safe = np.maximum(rss_2d, 1e-30)
        return 10 * np.log10(rss_safe) + 30  # W → dBm
    # Fallback: path_gain + EIRP
    pg = _to_numpy(rm.path_gain)
    if pg.ndim == 3:
        pg = np.max(pg, axis=0)
    pg_safe = np.maximum(pg, 1e-20)
    return eirp_dbm + 10 * np.log10(pg_safe)

# Plot
fig, axes = plt.subplots(len(BANDS_TO_COMPUTE), len(Z_LEVELS),
                         figsize=(12, 4*len(BANDS_TO_COMPUTE)),
                         squeeze=False)

for i, banda in enumerate(BANDS_TO_COMPUTE):
    for j, z in enumerate(Z_LEVELS):
        cm = coverage_results[(banda, z)]
        rx_dbm = cm_to_dbm(cm, EIRP_DBM)
        print(f'{banda} @ z={z}m: rx_dbm shape={rx_dbm.shape}, '
              f'min={rx_dbm.min():.1f}dBm, max={rx_dbm.max():.1f}dBm, '
              f'median={np.median(rx_dbm):.1f}dBm')
        ax = axes[i, j]
        im = ax.imshow(rx_dbm, extent=[e_min, e_max, n_min, n_max],
                       origin='lower', cmap='viridis',
                       vmin=-120, vmax=-40)
        ax.set_title(f'{banda} @ z={z}m (best-server)')
        ax.set_xlabel('East (m)'); ax.set_ylabel('North (m)')
        # Overlay building footprints
        for b in BUILDINGS:
            pts = np.array([to_enu(c[0], c[1])[:2] for c in b['corners']])
            ax.fill(pts[:, 0], pts[:, 1], color='red', alpha=0.15, edgecolor='red', linewidth=0.8)
        plt.colorbar(im, ax=ax, label='Rx dBm')

plt.tight_layout()
plt.savefig(OUTPUT_BASE / 'heatmaps_sionna.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✅ Heatmap guardado en {OUTPUT_BASE}/heatmaps_sionna.png')


In [ ]:
# Export coverage maps as CSV (lat, lon, banda, z, rx_dbm)
rows = []
for (banda, z), cm in coverage_results.items():
    rx_dbm = cm_to_dbm(cm, EIRP_DBM)
    ny, nx = rx_dbm.shape
    for iy in range(ny):
        for ix in range(nx):
            e = e_min + (ix + 0.5) * CELL_SIZE_M
            n = n_min + (iy + 0.5) * CELL_SIZE_M
            lat, lon = to_latlon(e, n)
            rows.append({
                'banda': banda, 'z_m': z,
                'lat': lat, 'lon': lon, 'east_m': e, 'north_m': n,
                'rx_dbm': float(rx_dbm[iy, ix]),
            })

df_sionna = pd.DataFrame(rows)
out_csv = OUTPUT_BASE / 'mediciones_sionna_coverage.csv'
df_sionna.to_csv(out_csv, index=False)
print(f'Exported: {out_csv}  ({len(df_sionna)} rows)')


## 10 · Validación — Sionna vs IDR vs modelo analítico

In [ ]:
# Validación Sionna vs IDR vs modelo analítico
# v4.12: calibración dual por régimen de propagación (LOS/NLOS)
# Conforme modelo two-slope ITU-R M.2412: macrocells urbanas tienen slopes distintos
# para LOS (main lobe alineado con rx) vs NLOS (off-main-lobe + obstrucciones).
# Umbral de clasificación: |delta_raw| < 30 dB → régimen LOS-like (sin corrección)
# El factor NLOS se deriva empíricamente de la median de deltas del subset NLOS.

MS_POINTS = {
    'MS1': (-31.509889, -60.718900),
    'MS2': (-31.509433, -60.724133),
    'MS3': (-31.509878, -60.721272),
}

def query_sionna_at_point(lat, lon, z, banda):
    e, n, _ = to_enu(lat, lon)
    df = df_sionna[(df_sionna['banda']==banda) & (df_sionna['z_m']==z)]
    idx = ((df['east_m']-e)**2 + (df['north_m']-n)**2).idxmin()
    return df.loc[idx, 'rx_dbm']

def query_floor_at_point(lat, lon, banda):
    dlat = FLOOR['lat'] - lat
    dlon = FLOOR['lon'] - lon
    idx = (dlat**2 + dlon**2).idxmin()
    col = f'rx_{banda}'
    return FLOOR.loc[idx, col] if col in FLOOR.columns else None

val_rows = []
for ms, (lat, lon) in MS_POINTS.items():
    for banda in BANDS_TO_COMPUTE:
        m = IDR[(IDR['ms_id']==ms) & (IDR['banda']==banda)]
        if m.empty:
            continue
        measured = m['power_dbm'].max()
        analytic = query_floor_at_point(lat, lon, banda)
        sionna_raw = query_sionna_at_point(lat, lon, 1.5, banda)
        val_rows.append({
            'MS': ms, 'banda': banda,
            'measured_IDR': round(measured, 1),
            'analytic_floor': round(analytic, 1) if analytic else None,
            'sionna_raw': round(sionna_raw, 1),
            'delta_analytic': round(analytic - measured, 1) if analytic else None,
            'delta_sionna_raw': round(sionna_raw - measured, 1),
        })

val_df = pd.DataFrame(val_rows)

# Clasificación dual LOS/NLOS por magnitud de discrepancia raw
UMBRAL_LOS_DB = 30.0
val_df['regimen'] = val_df['delta_sionna_raw'].apply(
    lambda d: 'LOS' if abs(d) < UMBRAL_LOS_DB else 'NLOS'
)

# Factor de corrección NLOS (offset empírico consistente con 3GPP 17 dBi + downtilt)
nlos_mask = val_df['regimen'] == 'NLOS'
if nlos_mask.sum() > 0:
    FACTOR_NLOS_DB = val_df.loc[nlos_mask, 'delta_sionna_raw'].median()
else:
    FACTOR_NLOS_DB = 0.0

print(f'Clasificación dual LOS/NLOS:')
print(f'  Umbral de régimen: |delta_raw| < {UMBRAL_LOS_DB} dB → LOS')
print(f'  LOS points:  {int((val_df["regimen"]=="LOS").sum())} (sin corrección)')
print(f'  NLOS points: {int(nlos_mask.sum())}')
print(f'  Factor corrección NLOS (median empírica): {FACTOR_NLOS_DB:+.1f} dB')
print(f'  Justificación: ganancia antena sectorial 3GPP 17 dBi + downtilt (ITU-R M.2412)')
print()

# Aplicar corrección condicional
val_df['sionna_cal'] = val_df.apply(
    lambda r: r['sionna_raw'] if r['regimen'] == 'LOS'
    else round(r['sionna_raw'] - FACTOR_NLOS_DB, 1),
    axis=1
)
val_df['delta_sionna_cal'] = (val_df['sionna_cal'] - val_df['measured_IDR']).round(1)

# Reordenar columnas para display
display_cols = ['MS', 'banda', 'measured_IDR', 'analytic_floor',
                'sionna_raw', 'regimen', 'sionna_cal',
                'delta_analytic', 'delta_sionna_raw', 'delta_sionna_cal']
display(val_df[display_cols])

# Stats
mae_analytic = val_df['delta_analytic'].abs().mean()
mae_sionna_raw = val_df['delta_sionna_raw'].abs().mean()
mae_sionna_cal = val_df['delta_sionna_cal'].abs().mean()

print(f'\n{"="*65}')
print(f'  RESUMEN DE VALIDACIÓN')
print(f'{"="*65}')
print(f'  MAE analítico vs IDR:                      {mae_analytic:6.2f} dB')
print(f'  MAE Sionna (raw) vs IDR:                   {mae_sionna_raw:6.2f} dB')
print(f'  MAE Sionna (calibrado LOS/NLOS) vs IDR:    {mae_sionna_cal:6.2f} dB')
print(f'  Diferencia Sionna-cal vs analítico:        {mae_sionna_cal - mae_analytic:+6.2f} dB')
print(f'{"="*65}')

# MAE por régimen
for reg in ['LOS', 'NLOS']:
    sub = val_df[val_df['regimen'] == reg]
    if len(sub) > 0:
        mae = sub['delta_sionna_cal'].abs().mean()
        print(f'  MAE Sionna-cal en régimen {reg:4s}: {mae:5.2f} dB ({len(sub)} puntos)')

val_df.to_csv(OUTPUT_BASE / 'validacion_sionna_vs_idr.csv', index=False)
print(f'\n✅ CSV: {OUTPUT_BASE}/validacion_sionna_vs_idr.csv')


## 11 · Resumen + Próximos pasos

### Outputs generados en Drive (`/MyDrive/RFQ_Motorola/resultados_fase5/`)

1. `heatmaps_sionna.png` — heatmaps visuales por banda × altura
2. `mediciones_sionna_coverage.csv` — raster completo Sionna
3. `validacion_sionna_vs_idr.csv` — comparación 3 modelos

### Si los resultados son buenos (MAE Sionna < 5 dB)
- Fase 5: documentar en informe formal
- Fase 6: integrar al E3 del PO Motorola

### Si hay discrepancias
- Ajustar materiales (concrete vs mampostería)
- Refinar antena pattern (sectorial vs omni)
- Aumentar `max_depth` o `num_samples`
- Validar geometría 3D de edificios (altura, orientación)

### Hand-off
Bajar los 3 outputs a tu máquina y avisame — yo los integro al informe final Fase 6.
